In [0]:
from pyspark import pipelines as dp
import requests
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp

API_KEY = "cbcf5759b3ba91b715fd9ef2a629e678"
BASE_URL = "https://v3.football.api-sports.io"

@dp.materialized_view(
    comment="Bronze layer table containing teams data from Football API"
)
def teams_bronze():
    headers = {
        "x-apisports-key": API_KEY
    }

    current_season_year = ''
    if datetime.today().month <= 6:
        current_season_year = str(datetime.today().year - 2)
    else:
        current_season_year = str(datetime.today().year - 1)
    
    url = f"{BASE_URL}/teams?league=39&season={current_season_year}"
    
    response = requests.get(url, headers=headers)
    data = response.json()
    team_clean = []
    
    for item in data["response"]:
        team_id = item["team"]["id"]
        team_name = item["team"]["name"]
        founded = item["team"]["founded"]
        country_name = item["team"]["country"]
        team_clean.append({
            "team_id": team_id,
            "team_name": team_name,
            "founded": founded,
            "country_name": country_name
        })
    
    current_date = str(datetime.today().strftime('%Y-%m-%d'))
    output_path = f"/Volumes/main/db_project_bronze/db_project_vol/teams_clean{current_date}.json"
    
    with open(output_path, "w") as f:
        json.dump(team_clean, f, indent=1)
    
    df = spark.read.json(output_path, multiLine=True)
    df = df.withColumn("insert_timestamp", current_timestamp())
    
    return df